In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

In [ ]:
# Importing all the libraries :

import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

print(" ALL the libraries have been imported !!...")

 ALL the libraries have been imported !!...


In [ ]:
# Setting up the groq api key

API_KEY = "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"

client=Groq(api_key=API_KEY)

In [ ]:
df = pd.read_csv('/content/college_notes.csv')

df.head()



,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...


In [ ]:
# Print the subjects:

df['subject'].value_counts()

,count
subject,
Data Engineering,5
Machine Learning,5
Generative AI,3
Python Programming,2


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   note_id  15 non-null     object
 1   subject  15 non-null     object
 2   topic    15 non-null     object
 3   content  15 non-null     object
dtypes: object(4)
memory usage: 612.0+ bytes


In [ ]:
df.shape

(15, 4)

In [ ]:
df.isnull().sum()

,0
note_id,0
subject,0
topic,0
content,0


In [ ]:
df

,note_id,subject,topic,content
0,N001,Data Engineering,ETL Pipelines,ETL stands for Extract Transform Load. It is t...
1,N002,Data Engineering,SQL Databases,A database is an organized collection of data ...
2,N003,Data Engineering,Data Cleaning,Data cleaning involves fixing or removing inco...
3,N004,Data Engineering,APIs and Data Collection,An API or Application Programming Interface al...
4,N005,Data Engineering,Big Data and PySpark,Big Data refers to extremely large datasets th...
5,N006,Machine Learning,Supervised Learning,Supervised learning is a type of machine learn...
6,N007,Machine Learning,Model Evaluation,Model evaluation measures how well a machine l...
7,N008,Machine Learning,Feature Engineering,Feature engineering is the process of selectin...
8,N009,Machine Learning,Decision Trees,A decision tree is a machine learning model th...
9,N010,Machine Learning,Random Forest,Random Forest is an ensemble learning method t...


In [ ]:
df['content_length']=df['content'].str.len()
df[['topic','content_length']]

,topic,content_length
0,ETL Pipelines,216
1,SQL Databases,209
2,Data Cleaning,210
3,APIs and Data Collection,224
4,Big Data and PySpark,242
5,Supervised Learning,255
6,Model Evaluation,240
7,Feature Engineering,236
8,Decision Trees,227
9,Random Forest,238


## Converting into chunks

In [ ]:
documents = df['content']

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]  # ids unique identifier for each document # Chroma db requires unique id # we convert note id into note_1,note_2,etc..


metadatas = [
    {"subject": row['subject'],"topic": row['topic']} for row in df.to_dict('records')
]

# print the essentials

print("Total chunks :",len(documents))
print("First document id :",ids[0])
print("First document metadata :",metadatas[0])
print()
print("First 200 letters of the document : \n" , documents[0][:200])

Total chunks : 15
First document id : note_N001
First document metadata : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}

First 200 letters of the document : 
 ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehou


## Embeddings

In [ ]:
#Loading the embedding model

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

test_embedding = embedding_model.encode("This is a RAG systems pipeline demo project....!!")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print(test_embedding)

[-8.76548886e-02  5.93985617e-02  3.00358776e-02 -4.83438522e-02
 -6.31509572e-02 -7.77551383e-02 -8.45739394e-02  2.69897282e-02
 -1.12020187e-01 -2.77695507e-02 -6.24415502e-02  1.49312261e-02
  4.87433188e-02 -4.55439501e-02 -8.76034200e-02  5.51527515e-02
  3.62206921e-02  1.15472339e-01 -3.48714255e-02 -4.32844758e-02
  1.29932491e-02  3.03999800e-02 -4.44415025e-02 -1.43429646e-02
 -1.24437045e-02  5.54168597e-02 -8.08866248e-02  3.14213112e-02
  7.22202612e-03 -8.42659175e-02 -9.83605627e-03 -2.55802199e-02
 -4.18359004e-02 -7.44129568e-02  1.06536515e-01 -4.60815057e-03
  6.55811206e-02  3.60408202e-02 -1.92540754e-02  1.61258616e-02
  1.05860205e-02 -7.89556056e-02  1.85609031e-02 -5.08846305e-02
  1.99549347e-02 -1.94796864e-02 -2.99556982e-02 -1.17564343e-01
 -1.86879355e-02  1.75941754e-02 -2.55081244e-02 -9.46916640e-02
  1.39961867e-02  1.32516753e-02 -4.33639884e-02 -5.09866104e-02
  1.92084648e-02 -7.45245069e-02 -4.98664975e-02 -1.77304223e-02
 -1.08626690e-02 -1.79879

In [ ]:
print(test_embedding.shape)

(384,)


## Chroma DB

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name = "College_Notes_Rag")

print("Chroma client created sucessfully !!")

Chroma client created sucessfully !!


### Convert the document (chunks) into embeddings

In [ ]:
embeddings = embedding_model.encode(documents.tolist(), show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## Adding documents into chromaDB

In [ ]:
embeddings_list = embeddings.tolist()


## ADD into chroma database
collection.add(
    documents = documents.tolist(),
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings_list
)

In [ ]:
collection

Collection(name=College_Notes_Rag)

In [ ]:
#Print the chromadb database
print(f"Number of documents in the collection: {collection.count()}")

Number of documents in the collection: 15


# Retrival - Searching the vector Database

In [ ]:
def retrive_relavant_chunks(question,top_k=3):

  """
  Given an user question, retrive relavent texts from chromaDB
  Parameters = question (str) , top_k (int)

  returns :
  A list of relavent chunks (str)
  """

  question_embedding = embedding_model.encode(question).tolist()  # Convert question into embedding

  # Now store that in the chromadb

  results = collection.query(
      query_embeddings = question_embedding,
      n_results = top_k,
      include = ["documents","metadatas","distances"]
  )

  return results

print("Retrival function defined sucessfully")

Retrival function defined sucessfully


In [ ]:
question = "What is RAG?"
results = retrive_relavant_chunks(question, top_k=3)

# Printing the retrived chunks
print("Top 3 retrived chunks:")
print("=="*30)

for i, (doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0],

)):
  print(f"\nResult {i+1}:")
  print(f"Subject : {meta['subject']}")
  print(f"Topic : {meta['topic']}")
  print(f"Distance : {dist}")
  print(f"Content : {doc[:200]}")

Top 3 retrived chunks:

Result 1:
Subject : Generative AI
Topic : Retrieval Augmented Generation
Distance : 1.0052646398544312
Content : RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This re

Result 2:
Subject : Data Engineering
Topic : Data Cleaning
Distance : 1.702364206314087
Content : Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted data. Common cleaning tasks include handling missing values removing duplicates fixing data types and standardizin

Result 3:
Subject : Data Engineering
Topic : ETL Pipelines
Distance : 1.7835520505905151
Content : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehou


# Context Injection in RAG (Retrieval-Augmented Generation)

---

## **What is Context Injection?**
**Context injection** in RAG is the process of **inserting retrieved documents** (or relevant information) directly into the LLM's prompt.
This allows the model to use **external, up-to-date, or domain-specific knowledge** to generate more accurate and context-aware responses.

---

## **How It Works: Step-by-Step**

### **1. Retrieval**
- User asks a question.
- System searches a **document database** (e.g., vector store, FAQs, knowledge base).
- Most relevant documents/chunks are **retrieved**.

### **2. Context Injection (Prompt Engineering)**
- Retrieved documents are **inserted into the LLM's prompt** as context.
- A **prompt template** is used to structure the input:
  ```text
  Answer the question using only the following context:

  ---
  Context:
  {retrieved_documents}
  ---

  Question: {user_question}

  Answer:

In [ ]:
def build_context_from_results(results):
    content_parts = []

    # Loop through documents and their metadata together
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        # Add subject line
        content_parts.append(f"Subject: {meta['subject']}")

        # Add source label with subject and topic
        chunk_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}]"
        content_parts.append(chunk_text)

        # Add the actual document text
        content_parts.append(doc)

    # Join everything into one big context string
    return "\n".join(content_parts)


In [ ]:
#Building the RAG generator function

def generate_rag_answer(user_question, top_k=3):
    """
    Given a user question, generate an answer using a RAG pipeline.

    """

    # 1. Retrieve relevant chunks
    retrieved_results = retrive_relavant_chunks(user_question, top_k=top_k)

    # 2. Build context from retrieved chunks
    context = build_context_from_results(retrieved_results)

    # 3. Define the system prompt and user prompt for the LLM
    system_prompt = "You are a helpful AI assistant."

    full_user_prompt = f"""Answer the question using only the following context:

---
Context:
{context}
---

Question: {user_question}

Answer:"""

    # 4. Generate the answer using the LLM (Groq client)
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": full_user_prompt,
            }
        ],
        model="llama-3.1-8b-instant", # Updated to a currently supported Groq model
        temperature=0.7,
        max_tokens=500,
    )

    # 5. Return the generated answer
    return chat_completion.choices[0].message.content

In [ ]:
user_question_1 = "What is RAG?"
answer_1 = generate_rag_answer(user_question_1)
print(f"Question: {user_question_1}")
print("--------------------------------------------------")
print(f"Answer: {answer_1}")

Question: What is RAG?
--------------------------------------------------
Answer: RAG stands for Retrieval Augmented Generation. It is a technique used in Generative AI where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This approach reduces hallucination and allows AI to answer questions about specific data.


In [ ]:
question = "What is RAG?"
results = retrive_relavant_chunks(question, top_k=3)

for i , (doc,meta) in enumerate(zip(results['documents'][0],results['metadatas'][0])):
  print(  f"\nResult {i+1}:")
  print(f"Subject : {meta['subject']}")
  print(f"Topic : {meta['topic']}")
  print(f"Content : {doc[:200]}")

NameError: name 'results' is not defined